# Records from PhysioNet

PhysioNet publishes its databases at
`https://physionet.org/files/<database>/<version>/`. To read a record from
there, set that URL as the **input directory** and turn **network access**
on. Network access is off by default.

This notebook reads record 100 of the MIT-BIH Arrhythmia Database (`mitdb`,
version 1.0.0, about 2 MB). Without a network connection, it reads the
sample record `100s` instead: the first minute of the same record.

In [ ]:
import numpy as np
import wfdbx

PHYSIONET = "https://physionet.org/files/mitdb/1.0.0/"

# Read from PhysioNet: set the input directory and turn network access on.
wfdbx.wfdbx_set_input_dir(PHYSIONET)
wfdbx.wfdbx_set_network(1)
try:
    hdr = wfdbx.read_header("100")        # reads PHYSIONET + "100.hea"
    record = "100"
except IOError:
    print("Could not read record 100 from PhysioNet (no network connection?);")
    print("using the sample record 100s shipped with wfdb-x instead.")
    wfdbx.wfdbx_set_network(0)
    wfdbx.wfdbx_set_input_dir(wfdbx.sample_data_dir())
    hdr = wfdbx.read_header("100s")
    record = "100s"
print("input directory:", wfdbx.wfdbx_get_input_dir())

In [ ]:
rec = wfdbx.read_record(record, sampto="0:10", physical=True)
ann = wfdbx.read_annotations(record, "atr")
print(f"record {record}: {hdr.nsig} signals at {rec.fs} Hz, "
      f"{len(ann)} annotations")
print(rec.names, rec.signals[:3])

Remote files are read once and kept for the rest of the session. The
signal file that a header names is read from the same URL as the header. A
record name is read from the input directory, so `read_header("100")` reads
`https://physionet.org/files/mitdb/1.0.0/100.hea`.

To read from PhysioNet for one call only, without changing the settings:

```python
rec = wfdbx.read_record("100", input_dir="https://physionet.org/files/mitdb/1.0.0/",
                        network=True)
```

The command-line tools take the same settings from environment variables:

```
WFDBX_NETWORK=1 WFDBX_INPUT_DIR=https://physionet.org/files/mitdb/1.0.0/ wfdbx rdsamp -r 100 -t 0:5
```

This works for any PhysioNet database. Its `RECORDS` file lists the names
of its records. To work offline, download the files of a record into a
directory: its header, the signal files the header names, and its
annotation files. Then use that directory as the input directory.

In [ ]:
# Back to the defaults: no network access, and the current directory.
wfdbx.wfdbx_set_input_dir(None)
wfdbx.wfdbx_set_network(0)
print("network access:", wfdbx.wfdbx_get_network(),
      "- input directory:", wfdbx.wfdbx_get_input_dir())